In [ ]:
# ZINB Regression
import pandas as pd
import numpy as np
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
tb = pd.read_table('fish.dat.txt', header=None, sep=',', names=['nofish', 'livebait', 'camper', 'persons', 'child', 'xb', 'zg', 'count'])


camper = torch.FloatTensor(np.array(tb['camper'])).reshape(-1, 1)
child = torch.FloatTensor(np.array(tb['child'])).reshape(-1, 1) 

X_train = torch.cat([camper, child], -1)

z_train = torch.FloatTensor(np.array(tb['persons'])).reshape(-1, 1)

y_train = torch.FloatTensor(np.array(tb['count'])).reshape(-1, 1)

In [ ]:
n, k  = X_train.shape
_, m = z_train.shape

In [ ]:
alpha = torch.FloatTensor([1])
beta = torch.zeros(k)
mu = []
for i in range(n):
    mu.append(torch.exp(beta.matmul(X_train[i])))
gamma = torch.zeros(m)

Lambda = []
for i in range(n):
    Lambda.append(torch.exp(gamma.matmul(z_train[i])))
pi = []
for i in range(n):
    pi.append(Lambda[i]/(1+Lambda[i]))

In [ ]:
learning_rate = 0.0001
epochs = 1000
optimizer = optim.SGD([alpha, beta, gamma], lr = learning_rate)

In [ ]:
for epoch in range(1, epochs+1):
    optimizer.zero_grad()

    y = y_train
    v = 1/alpha
    dLdBeta = []
    for r in range(k):
        sum_i = torch.FloatTensor([0])
        for i in range(len(y)):
            if y[i] == 0:
                sum_i = sum_i + ((-mu[i]*(1+alpha*mu[i])**(-1-v))/((Lambda[i]+(1+alpha*mu[i])**(-v))))*X_train[i][r]
        for i in range(len(y)):
            if y[i] > 0:
                sum_i = sum_i + ((y[i] - mu[i])/(1+alpha*mu[i])) * X_train[i][r]
        dLdBeta.append(-sum_i)
    beta.grad = torch.FloatTensor(dLdBeta)

    v = 1/alpha
    dLdGamma = []
    for r in range(m):
        sum_i = torch.FloatTensor([0])
        for i in range(len(y)):
            if y[i] == 0:
                sum_i = sum_i + (Lambda[i] / (Lambda[i] + (1+alpha*mu[i])**(-v)))*z_train[i][r]
        for i in range(n):
            sum_i = sum_i - (Lambda[i]/(1+Lambda[i]))*z_train[i][r]
        dLdGamma.append(-sum_i)

    gamma.grad = torch.FloatTensor(dLdGamma)

    v = 1/alpha
    dLdAlpha = torch.FloatTensor([0])
    for i in range(len(y)):
        if y[i] == 0:
            dLdAlpha += ((1+alpha*mu[i])*np.log(1+alpha*mu[i]) - alpha*mu[i])/((alpha**2 * (1+alpha*mu[i]) * (Lambda[i]*(1+alpha*mu[i])**v+1)))
    for i in range(len(y)):
        if y[i] > 0 :
            for j in range(int(y[i]-1)):
                dLdAlpha = dLdAlpha + -1/(alpha**2*j+alpha)
                dLdAlpha = dLdAlpha + np.log(1+alpha*mu[i])/alpha**2
                dLdAlpha = dLdAlpha + (y[i]-mu[i])/(alpha*(1+alpha*mu[i]))

    alpha.grad = torch.FloatTensor(-dLdAlpha)

    optimizer.step()
    mu = []
    for i in range(n):
        mu.append(torch.exp(beta.matmul(X_train[i])))
    Lambda = []
    for i in range(n):
        Lambda.append(torch.exp(gamma.matmul(z_train[i])))
    if epoch % 100 == 0:
        print(alpha, beta, gamma)
